# Text-to-Outfit — Notebook 1 (v3): pairing + filtros de calidad + caché para fine-tuning

**Novedades respecto a v2:**
- **Capa 1**: filtro de spam/afiliación (nombres de tienda, patrones promocionales, tallas).
- **Capa 4**: cap de frecuencia por etiqueta (solo en train) para evitar que unas pocas
  etiquetas muy repetidas dominen el entrenamiento.
- **Caché para fine-tuning**: además del `outfit_emb` congelado de siempre, ahora también se
  guarda la secuencia de salida justo *antes* de la última capa del Transformer
  (`prelast_seq` + `key_padding_mask`), para poder reentrenar solo esa última capa en el
  Notebook 2 sin volver a tocar imágenes.

Autocontenido: no hace falta pegar nada de `Recommendation_Engine_v2_accelerated.ipynb`.
Ajusta `POLYVORE_PATH` en la Sección 0.


## 0. Imports y configuración

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import os, json, random, re
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

POLYVORE_PATH = "./polyvore_kaggle/polyvore_outfits/"  # AJUSTA A TU RUTA LOCAL
IMAGES_DIR    = POLYVORE_PATH + "images/"
OUTPUT_DIR    = POLYVORE_PATH + "data/processed/"

TEXT_EMBED_DIM = 100
IMG_SIZE       = 224
MAX_LEN        = 8

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

## 1. Embeddings de texto precalculados por ítem

In [ ]:
TEXT_EMBEDDINGS_PATH = OUTPUT_DIR + "glove_text_embeddings.npy"
TEXT_INDEX_PATH      = OUTPUT_DIR + "glove_text_embeddings_index.json"

text_matrix = np.load(TEXT_EMBEDDINGS_PATH)
with open(TEXT_INDEX_PATH) as f:
    text_index = json.load(f)

assert text_matrix.shape[1] == TEXT_EMBED_DIM

item_id2textvec = {v: text_matrix[int(k)].astype(np.float32) for k, v in text_index.items()}
print(f"Embeddings de texto cargados: {len(item_id2textvec)} ítems, dim={text_matrix.shape[1]}")

## 2. Metadatos por ítem: imagen, categoría

In [ ]:
df = pd.read_csv(OUTPUT_DIR + "glove_text_features.csv")
df['item_id'] = df['item_id'].astype(str)

cat_encoder = LabelEncoder()
df['cat_id'] = cat_encoder.fit_transform(df['sub_category'])
NUM_CATEGORIES = df['cat_id'].nunique()
print(f"Categorías: {NUM_CATEGORIES}")

item_id2img_path = {iid: os.path.join(IMAGES_DIR, f"{iid}.jpg") for iid in df['item_id']}
item_id2cat      = dict(zip(df['item_id'], df['cat_id']))

missing_text = [iid for iid in df['item_id'] if iid not in item_id2textvec]
print(f"Ítems sin vector de texto: {len(missing_text)}")

## 3. Arquitectura — Stage 1: encoder multimodal end-to-end

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out = out + residual
        return self.relu(out)


def make_layer(in_planes, planes, num_blocks, stride=1):
    downsample = None
    if stride != 1 or in_planes != planes * BasicBlock.expansion:
        downsample = nn.Sequential(
            nn.Conv2d(in_planes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(planes * BasicBlock.expansion),
        )
    layers = [BasicBlock(in_planes, planes, stride=stride, downsample=downsample)]
    in_planes = planes * BasicBlock.expansion
    for _ in range(1, num_blocks):
        layers.append(BasicBlock(in_planes, planes))
    return nn.Sequential(*layers), in_planes


class SmallResNet18(nn.Module):
    def __init__(self, embedding_size=64):
        super().__init__()
        self.conv1   = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1     = nn.BatchNorm2d(64)
        self.relu    = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        in_planes = 64
        self.layer1, in_planes = make_layer(in_planes, 64,  num_blocks=2, stride=1)
        self.layer2, in_planes = make_layer(in_planes, 128, num_blocks=2, stride=2)
        self.layer3, in_planes = make_layer(in_planes, 256, num_blocks=2, stride=2)

        self.avgpool  = nn.AvgPool2d(kernel_size=7)
        self.flatten  = nn.Flatten()
        self.fc_embed = nn.Linear(256 * 2 * 2, embedding_size)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        return self.fc_embed(x)


class TextEncoder(nn.Module):
    def __init__(self, embedding_size=64, input_dim=TEXT_EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(input_dim, embedding_size)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, text_vec):
        return self.relu(self.proj(text_vec))


class CategoricalEncoder(nn.Module):
    def __init__(self, num_categories, embedding_size=64):
        super().__init__()
        self.embedding = nn.Embedding(num_categories, embedding_size)

    def forward(self, cat_ids):
        return self.embedding(cat_ids)


class ItemEncoder(nn.Module):
    def __init__(self, num_categories, common_dim=128):
        super().__init__()
        self.image_encoder = SmallResNet18(embedding_size=common_dim)
        self.text_encoder  = TextEncoder(embedding_size=common_dim)
        self.cat_encoder   = CategoricalEncoder(num_categories, embedding_size=common_dim)
        self.norm = nn.LayerNorm(common_dim, eps=1e-6)

    def forward(self, images, text_vecs, cat_ids):
        img_emb = self.image_encoder(images)
        txt_emb = self.text_encoder(text_vecs)
        cat_emb = self.cat_encoder(cat_ids)
        item_emb = img_emb + txt_emb + cat_emb
        return self.norm(item_emb)

## 4. Arquitectura — TAE (Type-Aware Gate)

In [ ]:
class TypeAwareGate(nn.Module):
    def __init__(self, num_categories, d_model):
        super().__init__()
        self.type_embedding = nn.Embedding(num_categories, d_model)
        self.gamma_proj = nn.Linear(d_model, d_model)
        self.beta_proj  = nn.Linear(d_model, d_model)

    def forward(self, x, cat_ids):
        t = self.type_embedding(cat_ids)
        gamma = torch.sigmoid(self.gamma_proj(t))
        beta  = self.beta_proj(t)
        return x * gamma + beta

## 5. Arquitectura — Stage 2: Transformer + modelo completo

**Modificado respecto a versiones anteriores:** se añade `encode_outfit_prelast(...)`, que
ejecuta todo el pipeline (ítems -> TAE -> `cls_token` -> todas las capas del Transformer
**excepto la última**) y devuelve la secuencia intermedia + la máscara de padding. `encode_outfit`
se reescribe para reutilizar este método internamente, así que su resultado sigue siendo
exactamente el mismo de siempre -- solo se expone el punto de corte.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads,
                                                dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm1(x + self.drop1(attn_out))
        ffn_out = self.ffn(x)
        return self.norm2(x + self.drop2(ffn_out))


class TAETransformerOutfitModel(nn.Module):
    def __init__(self, num_categories, d_model=128, n_heads=4, ffn_dim=256,
                 n_layers=2, dropout=0.1, max_len=MAX_LEN):
        super().__init__()
        self.max_len = max_len
        self.item_encoder = ItemEncoder(num_categories, common_dim=d_model)
        self.type_gate     = TypeAwareGate(num_categories, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
        ])
        self.dropout = nn.Dropout(dropout)

        self.ce_head = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 1), nn.Sigmoid(),
        )
        self.fitb_head = nn.Sequential(
            nn.Linear(d_model * 2, 128), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def encode_items(self, images, text_vecs, cat_ids):
        B, L = images.shape[0], images.shape[1]
        images_flat = images.reshape(B * L, 3, IMG_SIZE, IMG_SIZE)
        text_flat   = text_vecs.reshape(B * L, TEXT_EMBED_DIM)
        cats_flat   = cat_ids.reshape(B * L)

        item_emb = self.item_encoder(images_flat, text_flat, cats_flat)
        item_emb = self.type_gate(item_emb, cats_flat)
        item_emb = item_emb.reshape(B, L, -1)
        return item_emb

    def encode_outfit_prelast(self, images, text_vecs, cat_ids, padding_mask=None):
        """Ejecuta todo excepto la ÚLTIMA capa del Transformer. Devuelve la secuencia
        intermedia (incluye el cls_token en la posición 0) y la máscara de padding lista
        para usar directamente con esa última capa."""
        x = self.encode_items(images, text_vecs, cat_ids)
        x = self.dropout(x)

        batch_size = x.shape[0]
        cls = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)

        if padding_mask is not None:
            cls_mask = torch.zeros(batch_size, 1, dtype=torch.bool, device=x.device)
            key_padding_mask = torch.cat([cls_mask, padding_mask], dim=1)
        else:
            key_padding_mask = None

        for layer in self.encoder_layers[:-1]:
            x = layer(x, key_padding_mask=key_padding_mask)

        return x, key_padding_mask

    def encode_outfit(self, images, text_vecs, cat_ids, padding_mask=None):
        """Comportamiento idéntico al de siempre -- ahora implementado como
        encode_outfit_prelast(...) + la última capa, para no duplicar lógica."""
        x, key_padding_mask = self.encode_outfit_prelast(images, text_vecs, cat_ids, padding_mask)
        x = self.encoder_layers[-1](x, key_padding_mask=key_padding_mask)
        cls_out = F.normalize(x[:, 0, :], p=2, dim=-1)
        return cls_out

    def call_ce(self, images, text_vecs, cat_ids, padding_mask=None):
        cls = self.encode_outfit(images, text_vecs, cat_ids, padding_mask)
        return self.ce_head(cls)

## 6. Pipeline de datos — carga de imágenes y construcción de tensores de outfit

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def load_image(path):
    img = Image.open(path).convert("RGB")
    return image_transform(img)


def get_item_raw(item_id):
    item_id = str(item_id)
    return item_id2img_path[item_id], item_id2textvec[item_id], item_id2cat[item_id]


def build_outfit_tensors(item_ids, max_len=MAX_LEN):
    n = min(len(item_ids), max_len)
    images, texts, cats = [], [], []
    for iid in item_ids[:n]:
        path, text_vec, cat_id = get_item_raw(iid)
        images.append(load_image(path))
        texts.append(torch.from_numpy(text_vec))
        cats.append(cat_id)

    pad_needed = max_len - n
    images += [torch.zeros(3, IMG_SIZE, IMG_SIZE)] * pad_needed
    texts  += [torch.zeros(TEXT_EMBED_DIM)] * pad_needed
    cats   += [0] * pad_needed
    mask    = [False] * n + [True] * pad_needed

    images_t = torch.stack(images)
    texts_t  = torch.stack(texts).float()
    cats_t   = torch.tensor(cats, dtype=torch.long)
    mask_t   = torch.tensor(mask, dtype=torch.bool)
    return images_t, texts_t, cats_t, mask_t

## 7. Instanciar el modelo

In [ ]:
D_MODEL  = 128
N_HEADS  = 4
FFN_DIM  = 256
N_LAYERS = 2
DROPOUT  = 0.1

model = TAETransformerOutfitModel(
    num_categories=NUM_CATEGORIES,
    d_model=D_MODEL, n_heads=N_HEADS, ffn_dim=FFN_DIM, n_layers=N_LAYERS,
    dropout=DROPOUT, max_len=MAX_LEN,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales: {n_params:,}")

---
## A partir de aquí: pairing, filtros de calidad y extracción de embeddings

### A. Cargar el checkpoint entrenado

In [ ]:
CKPT_PATH_CE = POLYVORE_PATH + "checkpoints_RE/best_ce.pt"  # ajusta si hace falta

ckpt = torch.load(CKPT_PATH_CE, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
for p in model.parameters():
    p.requires_grad = False  # todo congelado por ahora -- el Notebook 2 descongelará
                              # explícitamente solo la última capa para el fine-tuning

print(f"Checkpoint cargado (época {ckpt['epoch']}, métrica {ckpt['best_metric']:.4f})")

### B. Reconstruir `outfit_id -> [item_ids ordenados]` y su split

In [ ]:
def build_outfit_item_map(json_paths_with_split):
    outfit_items = {}
    outfit_split = {}
    for path, split in json_paths_with_split:
        with open(path) as f:
            outfits = json.load(f)
        for outfit in outfits:
            set_id = str(outfit["set_id"])
            items_sorted = sorted(outfit["items"], key=lambda it: it["index"])
            outfit_items[set_id] = [str(it["item_id"]) for it in items_sorted]
            outfit_split[set_id] = split
    return outfit_items, outfit_split


OUTFIT_SPLIT_PATHS = [
    (POLYVORE_PATH + "nondisjoint/train.json", "train"),
    (POLYVORE_PATH + "nondisjoint/valid.json", "valid"),
    (POLYVORE_PATH + "nondisjoint/test.json",  "test"),
]
outfit_items, outfit_split = build_outfit_item_map(OUTFIT_SPLIT_PATHS)
print(f"Outfits indexados desde train/valid/test.json: {len(outfit_items)}")

### C. Emparejar cada outfit con su texto -- con **Capa 1: filtro de spam/afiliación**

Descarta outfits cuyo texto contenga nombres de tiendas conocidas, patrones promocionales
(`off`, `coupon`, `sale`...) o patrones de talla (`plus size`, `size 12`...). Es un filtro
barato y de bajo riesgo -- solo elimina texto que no describe el outfit en absoluto, no toca
las etiquetas de estilo/categoría genuinas (esas se dejan para la Capa 4, que rebalancea en
vez de eliminar).

In [ ]:
OUTFIT_TITLES_PATH = POLYVORE_PATH + "polyvore_outfit_titles.json"  # ajusta si hace falta

with open(OUTFIT_TITLES_PATH) as f:
    outfit_titles_raw = json.load(f)


def clean_text_basic(s):
    return " ".join(s.strip().split())


def get_outfit_text(entry):
    title = clean_text_basic(entry.get("title", ""))
    if title:
        return title
    return clean_text_basic(entry.get("url_name", ""))


# --- Capa 1: patrones de spam / afiliación / listing de producto ---
STORE_PATTERN = re.compile(
    r"\b(yesstyle|banggood|romwe|shein|zaful|aliexpress|paoloshoes|dhgate|rosegal|wish)\b",
    re.I,
)
PROMO_PATTERN = re.compile(r"\b(off|coupon|code|sale|discount|deal|free shipping)\b", re.I)
SIZE_PATTERN  = re.compile(r"\b(plus size|size \d|us \d|eu \d)\b", re.I)


def is_spam_text(text):
    return bool(STORE_PATTERN.search(text) or PROMO_PATTERN.search(text) or SIZE_PATTERN.search(text))


t2o_samples = []
skipped_no_text = 0
skipped_no_items = 0
skipped_missing_item_data = 0
skipped_spam = 0

for set_id, entry in outfit_titles_raw.items():
    text = get_outfit_text(entry)
    if not text:
        skipped_no_text += 1
        continue

    if is_spam_text(text):
        skipped_spam += 1
        continue

    item_ids = outfit_items.get(set_id)
    if not item_ids:
        skipped_no_items += 1
        continue

    if not all(iid in item_id2img_path and iid in item_id2textvec and iid in item_id2cat
               for iid in item_ids):
        skipped_missing_item_data += 1
        continue

    t2o_samples.append({
        "set_id": set_id,
        "item_ids": item_ids,
        "text": text,
        "split": outfit_split.get(set_id, "unknown"),
    })

print(f"Muestras Text-to-Outfit utilizables: {len(t2o_samples)}")
print(f"  Descartadas por falta de texto: {skipped_no_text}")
print(f"  Descartadas por spam/afiliación (Capa 1): {skipped_spam}")
print(f"  Descartadas por no encontrar el outfit en train/valid/test.json: {skipped_no_items}")
print(f"  Descartadas por falta de datos de algún ítem: {skipped_missing_item_data}")

t2o_train = [s for s in t2o_samples if s["split"] == "train"]
t2o_valid = [s for s in t2o_samples if s["split"] == "valid"]
t2o_test  = [s for s in t2o_samples if s["split"] == "test"]
print(f"Antes del cap de frecuencia -> train={len(t2o_train)}  valid={len(t2o_valid)}  test={len(t2o_test)}")

### C2. **Capa 4: cap de frecuencia por etiqueta** (solo sobre `train`)

Ninguna etiqueta única puede aportar más de `FREQ_CAP` outfits al entrenamiento -- si tiene
más, se muestrea aleatoriamente ese número. Aplicado **solo a `t2o_train`**: `valid`/`test` se
dejan intactos porque para evaluar sí queremos el corpus de candidatos completo y realista.

In [ ]:
# FREQ_CAP = 200

# def apply_frequency_cap(samples, cap=FREQ_CAP, seed=SEED):
#     rng = random.Random(seed)
#     groups = defaultdict(list)
#     for s in samples:
#         groups[s["text"]].append(s)

#     capped = []
#     n_capped_groups = 0
#     for text, group in groups.items():
#         if len(group) > cap:
#             capped.extend(rng.sample(group, cap))
#             n_capped_groups += 1
#         else:
#             capped.extend(group)
#     return capped, n_capped_groups


# t2o_train, n_capped = apply_frequency_cap(t2o_train, cap=FREQ_CAP)
# print(f"Etiquetas recortadas por el cap (>{FREQ_CAP} apariciones): {n_capped}")
# print(f"train tras el cap: {len(t2o_train)}  (valid={len(t2o_valid)}  test={len(t2o_test)} sin cambios)")

### D. Extraer y cachear representaciones de outfit (congelado + prelast para fine-tuning)

Para cada outfit se guardan ahora **tres** cosas:
- `outfit_emb`: el vector final congelado de siempre (128-d), para poder seguir usando/comparando
  contra la versión 100% congelada.
- `prelast_seq`: la secuencia justo antes de la última capa del Transformer (incluye el
  `cls_token`), que es la entrada que usará el fine-tuning en el Notebook 2.
- `key_padding_mask`: la máscara de padding correspondiente, que la última capa necesita para
  ignorar el relleno de outfits con menos de `MAX_LEN` prendas.

In [ ]:
@torch.no_grad()
def compute_outfit_representations(item_ids):
    images, texts, cats, mask = build_outfit_tensors(item_ids, max_len=MAX_LEN)
    images = images.unsqueeze(0).to(device)
    texts  = texts.unsqueeze(0).to(device)
    cats   = cats.unsqueeze(0).to(device)
    mask   = mask.unsqueeze(0).to(device)

    prelast_seq, key_padding_mask = model.encode_outfit_prelast(images, texts, cats, mask)
    last_out = model.encoder_layers[-1](prelast_seq, key_padding_mask=key_padding_mask)
    outfit_emb = F.normalize(last_out[:, 0, :], p=2, dim=-1)

    return (outfit_emb.squeeze(0).cpu(),
            prelast_seq.squeeze(0).cpu(),
            key_padding_mask.squeeze(0).cpu())


model.eval()
all_samples = t2o_train + t2o_valid + t2o_test
for s in tqdm(all_samples, desc="Extrayendo representaciones de outfit"):
    outfit_emb, prelast_seq, key_padding_mask = compute_outfit_representations(s["item_ids"])
    s["outfit_emb"] = outfit_emb
    s["prelast_seq"] = prelast_seq
    s["key_padding_mask"] = key_padding_mask

T2O_CACHE_PATH = POLYVORE_PATH + "checkpoints_RE/t2o_samples_with_embeddings_filtered_solocapauno.pt"
torch.save({"train": t2o_train, "valid": t2o_valid, "test": t2o_test}, T2O_CACHE_PATH)
print(f"Guardado: {T2O_CACHE_PATH}")

---
### Siguiente paso

Continúa con `text_to_outfit_02_query_encoder_training_v3.ipynb`, que carga este `.pt` (ahora
con `train`/`valid`/`test` ya separados dentro del propio fichero) y entrena conjuntamente el
`QueryEncoder` **y** la última capa del Transformer (fine-tuning), con la pérdida contrastiva
multi-positivo.